# ДЗ-2, опыт 15: SwiGLU вместо ReLU в Pre-LN Transformer

Один архитектурный фактор: заменить FFN во всех 2 encoder + 2 decoder слоях. Проверяем, улучшит ли управляемое входом произведение признаков качество генерируемых программ при почти том же числе параметров.

- ReLU: $\mathrm{ReLU}(xW_1+b_1)W_2+b_2$, внутренняя ширина 1024.
- SwiGLU: $(\mathrm{SiLU}(xW_g+b_g)\odot(xW_v+b_v))W_o+b_o$, ширина 680, $\mathrm{SiLU}(z)=z\sigma(z)$.
- $x$ — представление одного токена после attention и Pre-LN; $\odot$ — поэлементное произведение. Ветвь gate управляет вкладом value в зависимости от входа. Значения gate не обязаны лежать в $[0,1]$.
- Три матрицы вместо двух: $3dg\approx2dm$, поэтому $g\approx2m/3$. Всего 6 306 112 параметров против 6 312 960 у ReLU (−0,11%). Bias и dropout0.1 сохранены.
- Те же 100k train-пар, BPE4096 из опыта08, Pre-LN, batch8/FP32, KV-cache/greedy. Обучение **с нуля до 100000 шагов**, старый checkpoint загружать не нужно. Initial embeddings/attention/norms/head сохраняют seed42 инициализацию13; новые FFN инициализируются отдельно без сдвига RNG обучения.
- LR совпадает с 13+14: warmup1000 до0.001, cosine до0.0001 на50000; затем0.0001 до 100000. Горизонт cosine не растягивается.
- В50k/75k/100k сравниваем с сохранёнными результатами 14 на тех же 1024 heldout: token-weighted CE, фиксированные 888 арифметических reference, EOS/undefined names и время test1919. Numeric agreement — ограниченная диагностика; часть reference ошибочна, это не официальный pass@1.

Запусти в **новой Colab-сессии с T4** сверху вниз. Данные публичные; BPE и исторические сводки встроены. В конце скачай ZIP и этот IPYNB с outputs. Если сессия прервалась, выполни последнюю ячейку архива: она сохранит доступные результаты и latest checkpoint без повторного обучения. Ориентир прошлых запусков — несколько десятков минут; фактическую скорость SwiGLU измеряем здесь.

Источник архитектурной идеи: [Shazeer, GLU Variants Improve Transformer,2020](https://arxiv.org/abs/2002.05202). Улучшение на нашем кодовом датасете — гипотеза.

In [1]:
%pip install -q tokenizers==0.23.2 pyarrow


## 1. Настройки и отдельный каталог запуска

In [2]:
import ast
import base64
import io
import contextlib
import zlib
import hashlib
import json
import math
import os
import random
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 100000, 8
TOTAL_STEPS, SCHEDULE_TOTAL_STEPS = 100000, 50000
EVALUATION_STEPS = (2000, 5000, 10000, 20000, 30000, 50000, 75000, TOTAL_STEPS)
FULL_EVALUATION_STEPS = (50000, 75000, TOTAL_STEPS)
HELDOUT_SIZE = 1024
LOG_EVERY, CHECKPOINT_EVERY = 50, 1000
D_MODEL, N_LAYERS, MAX_POSITIONS = 256, 2, 2048
SWIGLU_DIM = 680  # 2/3 * 1024 rounded to a nearby multiple of8; almost equal parameter budget
LEARNING_RATE, MIN_LEARNING_RATE = 0.001, 0.0001
WARMUP_STEPS = 1000
PAD, BOS, EOS, TOKEN_OFFSET, TARGET_VOCAB = 0, 1, 2, 3, 4096
GENERATION_CAP, INFERENCE_BATCH, BYTE_BUDGET = 512, 32, 512
LOGIT_ATOL, LOGIT_RTOL = 1e-4, 1e-4
RUN_DIR = Path('hw02-swiglu-preln-bpe4k-100k-pairs-100000-steps-heldout1024')
RESULTS_FILE = RUN_DIR / 'results.json'
TOKENIZER_FILE = RUN_DIR / 'tokenizer.json'
LATEST_CHECKPOINT = RUN_DIR / 'latest.pt'
ARCHIVE_FILE = Path('hw02-swiglu-preln-bpe4k-100k-heldout1024-results.zip')
assert tokenizers.__version__ == '0.23.2', 'Run the installation cell in a fresh session'
assert torch.cuda.is_available(), 'Включи GPU в настройках Colab'
assert not RESULTS_FILE.exists() and not LATEST_CHECKPOINT.exists(), 'Сначала сохрани прежний запуск; используй новую сессию'
RUN_DIR.mkdir(exist_ok=True)
device = torch.device('cuda')
random.seed(SEED)
torch.manual_seed(SEED)

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()
def sha(raw): return hashlib.sha256(raw).hexdigest()
def data_hash(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode())
def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)
def save_results(): write_json(RESULTS_FILE, result)

result = {'experiment': '15-swiglu-preln-bpe4k-100k-pairs-100000-steps-heldout1024', 'phase': 'setup', 'actual_steps': 0,
    'runtime': {'device': str(device), 'gpu': torch.cuda.get_device_name(0),
                'torch': torch.__version__, 'tokenizers': tokenizers.__version__},
    'training_log': [], 'diagnostics': {}, 'checkpoint_files': {},
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0,
    'run_finished': False, 'test_evaluated': False, 'functional_quality_evaluated': False,
    'historical_relu14': {}}
result['historical_relu14'] = {'provenance': 'Saved user T4 run14, ReLU FFN; historical measurements, not new run15 results', 'source_results_sha256': '1d3b8fcc8e0449ae44848d2b854df1f132cac15ed6789eea15b6b5e09273f5ce', 'parameters': 6312960, 'config': {'seed': 42, 'train_size': 100000, 'batch_size_train': 8, 'total_steps': 100000, 'evaluation_steps': [50000, 75000, 100000], 'd_model': 256, 'encoder_layers': 2, 'decoder_layers': 2, 'heads': 4, 'ffn_dim': 1024, 'dropout': 0.1, 'activation': 'ReLU', 'norm_first': True, 'lr': 0.001, 'optimizer': 'AdamW', 'weight_decay': 0.01, 'clip_norm': 1.0, 'scheduler': {'name': 'LambdaLR', 'schedule': 'linear_warmup_then_cosine', 'total_steps': 50000, 'warmup_steps': 1000, 'cosine_steps': 49000, 'peak_lr': 0.001, 'eta_min': 0.0001, 'first_update_lr': 1e-06, 'restarts': False, 'step_unit': 'optimizer update', 'indexing': 's=1..T; updateW uses peak; updateT uses floor; scheduler index counts completed updates'}, 'dtype': 'float32', 'tokenizer': 'own byte-level BPE', 'vocab': 4096, 'content_id_offset': 3, 'special_ids': {'pad': 0, 'bos': 1, 'eos': 2}, 'tokenizer_sha256': '73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86', 'max_positions': 2048, 'generation_cap': 512, 'batch_size_inference': 32, 'inference': 'Pre-LN incremental KV cache', 'tokenizer_source_experiment': '08', 'training_continuation': {'initial_step': 50000, 'target_steps': 100000, 'additional_updates': 50000, 'lr_after_original_horizon': 0.0001, 'source_checkpoint_sha256': '3b91262115ed541a59977513a51c7f94144049c957c630dcb2e579efc1a46c7c'}}, 'data': {'revision': 'b2724689ff079e9c22efd548fe134bfb29036dbb', 'shard': 0, 'source_file_sha256': '91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec', 'training_examples': 100000, 'base20k_sha256': '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e', 'base20k_prefix_unchanged': True, 'train64_sha256': '125106742b04a01ebe70f106730c0bd9a30f52244bef86b4215f27591d200203', 'validation_examples': 64, 'total_heldout_examples': 1024, 'total_pair_presentations_planned': 800000, 'mean_pair_presentations_planned': 8.0, 'unique_train_questions': 99212, 'train_sha256': '7a141e91ce958206be7e18d3c50b5ccf72a10f1d94890c5f0c3927aa497e47b9', 'validation64_sha256': '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050', 'train_row_indices_sha256': 'fd49ac534874f1a49777ea2277b8c940c2a3fbea908ec05027bbe1245db34d86', 'extra_heldout_examples': 192, 'extra_heldout_sha256': 'a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc', 'extra_selection': 'first 192 distinct eligible hash-heldout questions after excluding original validation64', 'question_overlap_train_heldout': 0, 'download_and_selection_seconds': 7.602486843000008, 'selection': 'first100000 eligible non-heldout rows of shard0; original20000 unchanged prefix; question<=512/code<512 UTF8 bytes', 'new_heldout_examples': 768, 'new_heldout_sha256': 'e8273f55b276a9c5d73e18ff89bc8cbf550dbfec2a25380063635fa39d3fd3bd', 'heldout1024_sha256': 'e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743', 'heldout256_sha256': '6735359ee7a7e1b0738f8180d810b3aaff855307ec1b067c504e099295ff37f8', 'new_heldout_selection': 'next768 distinct eligible hash-heldout questions after original64+192; same pinned shard0', 'additional_pair_presentations': 400000}, 'controls': {'50000': {'heldout256': {'examples': 256, 'bpe_loss': 1.0089728666212037, 'numeric_audit': {'heldout_pairs_total': 256, 'reference_numeric_pairs': 223, 'reference_unsupported_pairs': 33, 'questions_and_references_sha256': '9dd311874ca6e5839ed8c9569d24584fe7965ec15eeefc28da39e18f7225a5f6', 'generated_statuses_on_fixed_reference_subset': {'numeric': 192, 'undefined_name': 25, 'syntax_error': 1, 'unsupported': 5}, 'numeric_reference_agreements': 24, 'wrong_numeric_count': 168, 'definite_failure_count': 26, 'unresolved_generated_unsupported_count': 5, 'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}, 'parses_as_python_count': 255, 'eos_emitted_count': 248, 'token_cap_without_eos_count': 8, 'undefined_name_count': 31, 'generation_seconds_sum': 8.513993373999938, 'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}, 'heldout1024': {'examples': 1024, 'bpe_loss': 1.0315946050342133, 'numeric_audit': {'heldout_pairs_total': 1024, 'reference_numeric_pairs': 888, 'reference_unsupported_pairs': 136, 'questions_and_references_sha256': '962724938ba673e6b613f62965766e27155bb01b5cb2d72b96b7807c8a18f179', 'generated_statuses_on_fixed_reference_subset': {'numeric': 762, 'undefined_name': 87, 'syntax_error': 3, 'unsupported': 30, 'arithmetic_error': 5, 'no_return': 1}, 'numeric_reference_agreements': 104, 'wrong_numeric_count': 658, 'definite_failure_count': 96, 'unresolved_generated_unsupported_count': 30, 'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}, 'parses_as_python_count': 1019, 'eos_emitted_count': 999, 'token_cap_without_eos_count': 25, 'undefined_name_count': 102, 'generation_seconds_sum': 28.82011988199997, 'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}, 'validation64_conditioning': {'examples': 64, 'correct_bpe_loss': 0.9943785713181157, 'rotated_questions_bpe_loss': 2.4703810210420167, 'rotated_minus_correct': 1.476002449723901, 'scope': 'fixed rotation by1; only source questions change, gold codes/masks/weights unchanged; eval mode; sensitivity, not pass@1'}, 'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 0.8737954400745949, 'seconds': 0.6848325890000524, 'answers_per_second': 93.4534965595732, 'generated_tokens_including_eos': 3575, 'generated_utf8_bytes': 12582, 'mean_tokens_including_eos': 55.859375, 'mean_utf8_bytes': 196.59375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 163.94287109375, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.0373491699999704, 'unique_generated_code_count': 64, 'most_common_generated_code_count': 1, 'most_common_generated_code_fraction': 0.015625, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 53, 'undefined_name': 10, 'syntax_error': 1}, 'numeric_ast_undefined_name_count': 10, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 2, 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 63, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 64, 'reached_token_cap_count': 0, 'token_cap_without_eos_count': 0, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 48, 'numeric_ast_match_count': 7, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5220.253909382409, 'output_bytes_per_second': 18372.373339258593}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 0.9943785713181157, 'seconds': 1.9343016939999416, 'answers_per_second': 33.08687584699077, 'generated_tokens_including_eos': 3991, 'generated_utf8_bytes': 13837, 'mean_tokens_including_eos': 62.359375, 'mean_utf8_bytes': 216.203125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 160.9140625, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.024131046000093193, 'unique_generated_code_count': 64, 'most_common_generated_code_count': 1, 'most_common_generated_code_fraction': 0.015625, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 54, 'undefined_name': 9, 'syntax_error': 1}, 'numeric_ast_undefined_name_count': 9, 'repeated_tail_32_plus_count': 1, 'consecutive_lines_3_plus_count': 2, 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 62, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 63, 'reached_token_cap_count': 1, 'token_cap_without_eos_count': 1, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 46, 'numeric_ast_match_count': 2, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 2063.27689852094, 'output_bytes_per_second': 7153.485954606426}}, '75000': {'heldout256': {'examples': 256, 'bpe_loss': 0.9662917785388481, 'numeric_audit': {'heldout_pairs_total': 256, 'reference_numeric_pairs': 223, 'reference_unsupported_pairs': 33, 'questions_and_references_sha256': '9dd311874ca6e5839ed8c9569d24584fe7965ec15eeefc28da39e18f7225a5f6', 'generated_statuses_on_fixed_reference_subset': {'numeric': 191, 'undefined_name': 27, 'unsupported': 3, 'no_return': 1, 'syntax_error': 1}, 'numeric_reference_agreements': 18, 'wrong_numeric_count': 173, 'definite_failure_count': 29, 'unresolved_generated_unsupported_count': 3, 'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}, 'parses_as_python_count': 255, 'eos_emitted_count': 249, 'token_cap_without_eos_count': 7, 'undefined_name_count': 32, 'generation_seconds_sum': 6.99509366999996, 'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}, 'heldout1024': {'examples': 1024, 'bpe_loss': 0.9894610480589706, 'numeric_audit': {'heldout_pairs_total': 1024, 'reference_numeric_pairs': 888, 'reference_unsupported_pairs': 136, 'questions_and_references_sha256': '962724938ba673e6b613f62965766e27155bb01b5cb2d72b96b7807c8a18f179', 'generated_statuses_on_fixed_reference_subset': {'numeric': 747, 'undefined_name': 104, 'unsupported': 24, 'no_return': 1, 'syntax_error': 7, 'arithmetic_error': 5}, 'numeric_reference_agreements': 97, 'wrong_numeric_count': 650, 'definite_failure_count': 117, 'unresolved_generated_unsupported_count': 24, 'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}, 'parses_as_python_count': 1016, 'eos_emitted_count': 1000, 'token_cap_without_eos_count': 24, 'undefined_name_count': 120, 'generation_seconds_sum': 25.49235849599995, 'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}, 'validation64_conditioning': {'examples': 64, 'correct_bpe_loss': 0.9642669619159369, 'rotated_questions_bpe_loss': 2.477455593116453, 'rotated_minus_correct': 1.5131886312005158, 'scope': 'fixed rotation by1; only source questions change, gold codes/masks/weights unchanged; eval mode; sensitivity, not pass@1'}, 'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 0.8203988040286033, 'seconds': 1.3229615949999243, 'answers_per_second': 48.37630982024362, 'generated_tokens_including_eos': 5096, 'generated_utf8_bytes': 15637, 'mean_tokens_including_eos': 79.625, 'mean_utf8_bytes': 244.328125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 207.0068359375, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.021307260000185124, 'unique_generated_code_count': 64, 'most_common_generated_code_count': 1, 'most_common_generated_code_fraction': 0.015625, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 13, 'numeric': 48, 'unsupported': 2, 'syntax_error': 1}, 'numeric_ast_undefined_name_count': 13, 'repeated_tail_32_plus_count': 2, 'consecutive_lines_3_plus_count': 1, 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 61, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 61, 'reached_token_cap_count': 3, 'token_cap_without_eos_count': 3, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 43, 'numeric_ast_match_count': 8, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3851.9636694368983, 'output_bytes_per_second': 11819.693072799211}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 0.9642669619159369, 'seconds': 1.291645883000001, 'answers_per_second': 49.54918437192119, 'generated_tokens_including_eos': 4293, 'generated_utf8_bytes': 14804, 'mean_tokens_including_eos': 67.078125, 'mean_utf8_bytes': 231.3125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 204.03759765625, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.019193868999991537, 'unique_generated_code_count': 64, 'most_common_generated_code_count': 1, 'most_common_generated_code_fraction': 0.015625, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 54, 'undefined_name': 7, 'unsupported': 3}, 'numeric_ast_undefined_name_count': 7, 'repeated_tail_32_plus_count': 1, 'consecutive_lines_3_plus_count': 3, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 63, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 63, 'reached_token_cap_count': 1, 'token_cap_without_eos_count': 1, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 46, 'numeric_ast_match_count': 2, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3323.666382947776, 'output_bytes_per_second': 11461.34571003002}}, '100000': {'heldout256': {'examples': 256, 'bpe_loss': 0.9366433386978137, 'numeric_audit': {'heldout_pairs_total': 256, 'reference_numeric_pairs': 223, 'reference_unsupported_pairs': 33, 'questions_and_references_sha256': '9dd311874ca6e5839ed8c9569d24584fe7965ec15eeefc28da39e18f7225a5f6', 'generated_statuses_on_fixed_reference_subset': {'numeric': 198, 'undefined_name': 19, 'unsupported': 5, 'syntax_error': 1}, 'numeric_reference_agreements': 30, 'wrong_numeric_count': 168, 'definite_failure_count': 20, 'unresolved_generated_unsupported_count': 5, 'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}, 'parses_as_python_count': 255, 'eos_emitted_count': 251, 'token_cap_without_eos_count': 5, 'undefined_name_count': 26, 'generation_seconds_sum': 6.976885899999843, 'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}, 'heldout1024': {'examples': 1024, 'bpe_loss': 0.9619496269115947, 'numeric_audit': {'heldout_pairs_total': 1024, 'reference_numeric_pairs': 888, 'reference_unsupported_pairs': 136, 'questions_and_references_sha256': '962724938ba673e6b613f62965766e27155bb01b5cb2d72b96b7807c8a18f179', 'generated_statuses_on_fixed_reference_subset': {'numeric': 771, 'undefined_name': 84, 'unsupported': 24, 'syntax_error': 2, 'arithmetic_error': 7}, 'numeric_reference_agreements': 117, 'wrong_numeric_count': 654, 'definite_failure_count': 93, 'unresolved_generated_unsupported_count': 24, 'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}, 'parses_as_python_count': 1020, 'eos_emitted_count': 1008, 'token_cap_without_eos_count': 16, 'undefined_name_count': 105, 'generation_seconds_sum': 22.780550751999954, 'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}, 'validation64_conditioning': {'examples': 64, 'correct_bpe_loss': 0.9332075503417031, 'rotated_questions_bpe_loss': 2.4727715841868103, 'rotated_minus_correct': 1.539564033845107, 'scope': 'fixed rotation by1; only source questions change, gold codes/masks/weights unchanged; eval mode; sensitivity, not pass@1'}, 'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 0.7774230942591894, 'seconds': 1.3398212520000925, 'answers_per_second': 47.76756593796407, 'generated_tokens_including_eos': 4805, 'generated_utf8_bytes': 15652, 'mean_tokens_including_eos': 75.078125, 'mean_utf8_bytes': 244.5625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 213.48583984375, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.020805172999871502, 'unique_generated_code_count': 64, 'most_common_generated_code_count': 1, 'most_common_generated_code_fraction': 0.015625, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 13, 'numeric': 48, 'unsupported': 2, 'syntax_error': 1}, 'numeric_ast_undefined_name_count': 13, 'repeated_tail_32_plus_count': 1, 'consecutive_lines_3_plus_count': 5, 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 62, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 62, 'reached_token_cap_count': 2, 'token_cap_without_eos_count': 2, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 46, 'numeric_ast_match_count': 8, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3586.299286436209, 'output_bytes_per_second': 11682.155344703338}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 0.9332075503417031, 'seconds': 0.7906489889999193, 'answers_per_second': 80.94616054711294, 'generated_tokens_including_eos': 3709, 'generated_utf8_bytes': 13237, 'mean_tokens_including_eos': 57.953125, 'mean_utf8_bytes': 206.828125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 210.046875, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.034824348000029204, 'unique_generated_code_count': 64, 'most_common_generated_code_count': 1, 'most_common_generated_code_fraction': 0.015625, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 56, 'undefined_name': 7, 'unsupported': 1}, 'numeric_ast_undefined_name_count': 7, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 2, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 64, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 64, 'reached_token_cap_count': 0, 'token_cap_without_eos_count': 0, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 47, 'numeric_ast_match_count': 4, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 4691.082960456904, 'output_bytes_per_second': 16741.942611908344}}}, 'test': {'split': 'test1919', 'examples': 1919, 'bpe_loss': None, 'seconds': 41.193087706999904, 'answers_per_second': 46.58548574094644, 'generated_tokens_including_eos': 129497, 'generated_utf8_bytes': 426667, 'mean_tokens_including_eos': 67.48150078165712, 'mean_utf8_bytes': 222.3381969775925, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 223.4833984375, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.39908324099997117, 'unique_generated_code_count': 1906, 'most_common_generated_code_count': 3, 'most_common_generated_code_fraction': 0.001563314226159458, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 1587, 'undefined_name': 237, 'unsupported': 65, 'syntax_error': 18, 'arithmetic_error': 10, 'no_return': 2}, 'numeric_ast_undefined_name_count': 237, 'repeated_tail_32_plus_count': 27, 'consecutive_lines_3_plus_count': 87, 'syntax_error_types': {'SyntaxError': 18}, 'parses_as_python_count': 1901, 'defines_solve_count': 1901, 'solve_returns_value_count': 1874, 'indentation_error_count': 0, 'valid_utf8_count': 1919, 'eos_emitted_count': 1884, 'reached_token_cap_count': 35, 'token_cap_without_eos_count': 35, 'byte_budget_stop_count': 0, 'bpe_tokens_per_second': 3143.6584924415533, 'output_bytes_per_second': 10357.732904967375, 'within_180_seconds': True}}
save_results()
print('RUNTIME', json.dumps(result['runtime']))


RUNTIME {"device": "cuda", "gpu": "Tesla T4", "torch": "2.11.0+cu130", "tokenizers": "0.23.2"}


## 2. Неизменные100000 train +1024 heldout

64+192+768 distinct heldout-вопросов, ни один не входит в train. Проверяем хеши и прежний порядок.

In [3]:
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
train_file = cache / 'train.parquet'
data_started = time.perf_counter()
if not train_file.exists(): urlretrieve(f'{base_url}/data/train/part-00000.parquet', train_file)
train_file_hash = sha(train_file.read_bytes())
assert train_file_hash == '91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec', 'Train shard checksum mismatch'
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

train, train_row_indices, validation = [], [], None
heldout_pool, heldout_questions = [], set()
offset = 0
for batch in pq.ParquetFile(train_file).iter_batches(batch_size=4096):
    rows = batch.to_pylist()
    if validation is None:
        validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
        heldout_questions.update(r['question'] for r in validation)
    for index, row in enumerate(rows):
        if not eligible(row): continue
        if is_validation(row['question']):
            if row['question'] not in heldout_questions and len(heldout_pool) < HELDOUT_SIZE - len(validation):
                heldout_pool.append(row)
                heldout_questions.add(row['question'])
            continue
        train.append(row)
        train_row_indices.append(offset + index)
        if len(train) == TRAIN_SIZE: break
    if len(train) == TRAIN_SIZE: break
    offset += len(rows)
assert len(train) == TRAIN_SIZE == 100000 and len(validation) == 64 and len(heldout_pool) == HELDOUT_SIZE - 64
train_hash, validation_hash = data_hash(train), data_hash(validation)
BASE20K_SHA256 = '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e'
assert data_hash(train[:20000]) == BASE20K_SHA256, 'Original20k prefix changed'
assert train_hash == '7a141e91ce958206be7e18d3c50b5ccf72a10f1d94890c5f0c3927aa497e47b9'
assert data_hash(train_row_indices) == 'fd49ac534874f1a49777ea2277b8c940c2a3fbea908ec05027bbe1245db34d86'
assert data_hash(train[:64]) == '125106742b04a01ebe70f106730c0bd9a30f52244bef86b4215f27591d200203'
assert data_hash(heldout_pool[:192]) == 'a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc'
assert len(set(r['question'] for r in train)) == 99212
assert data_hash(train[:512]) == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35'
assert validation_hash == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert not ({r['question'] for r in train} & heldout_questions)
assert all(not is_validation(r['question']) for r in train)
train_diagnostic = train[:64]
validation_extra = heldout_pool[:192]
validation_new = heldout_pool[192:]
heldout_all = validation + validation_extra + validation_new
assert len(heldout_all) == len({r['question'] for r in heldout_all}) == HELDOUT_SIZE
assert len(validation_new) == 768
result['data'] = {'revision': revision, 'shard': 0, 'source_file_sha256': train_file_hash,
    'training_examples': len(train), 'base20k_sha256': data_hash(train[:20000]),
    'base20k_prefix_unchanged': True, 'train64_sha256': data_hash(train_diagnostic),
    'validation_examples': len(validation), 'total_heldout_examples': len(heldout_all),
    'total_pair_presentations_planned': TOTAL_STEPS * BATCH_SIZE,
    'mean_pair_presentations_planned': TOTAL_STEPS * BATCH_SIZE / len(train),
    'unique_train_questions': len({r['question'] for r in train}),
    'train_sha256': train_hash, 'validation64_sha256': validation_hash,
    'train_row_indices_sha256': data_hash(train_row_indices),
    'extra_heldout_examples': len(validation_extra), 'extra_heldout_sha256': data_hash(validation_extra),
    'extra_selection': 'first 192 distinct eligible hash-heldout questions after excluding original validation64',
    'question_overlap_train_heldout': 0, 'download_and_selection_seconds': time.perf_counter() - data_started,
    'selection': 'first100000 eligible non-heldout rows of shard0; original20000 unchanged prefix; question<=512/code<512 UTF8 bytes'}
result['data'].update({'new_heldout_examples': len(validation_new),
    'new_heldout_sha256': data_hash(validation_new), 'heldout1024_sha256': data_hash(heldout_all),
    'heldout256_sha256': data_hash(validation + validation_extra),
    'new_heldout_selection': 'next768 distinct eligible hash-heldout questions after original64+192; same pinned shard0'})
result['phase'] = 'data_ready'
save_results()
print('DATA', json.dumps(result['data']))
assert data_hash(validation_new) == 'e8273f55b276a9c5d73e18ff89bc8cbf550dbfec2a25380063635fa39d3fd3bd'
assert data_hash(heldout_all) == 'e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743'
for key in ['train_sha256', 'validation64_sha256', 'extra_heldout_sha256',
            'new_heldout_sha256', 'heldout1024_sha256', 'heldout256_sha256']:
    assert result['data'][key] == result['historical_relu14']['data'][key], key


DATA {"revision": "b2724689ff079e9c22efd548fe134bfb29036dbb", "shard": 0, "source_file_sha256": "91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec", "training_examples": 100000, "base20k_sha256": "1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e", "base20k_prefix_unchanged": true, "train64_sha256": "125106742b04a01ebe70f106730c0bd9a30f52244bef86b4215f27591d200203", "validation_examples": 64, "total_heldout_examples": 1024, "total_pair_presentations_planned": 800000, "mean_pair_presentations_planned": 8.0, "unique_train_questions": 99212, "train_sha256": "7a141e91ce958206be7e18d3c50b5ccf72a10f1d94890c5f0c3927aa497e47b9", "validation64_sha256": "0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050", "train_row_indices_sha256": "fd49ac534874f1a49777ea2277b8c940c2a3fbea908ec05027bbe1245db34d86", "extra_heldout_examples": 192, "extra_heldout_sha256": "a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc", "extra_selection": "first 192 d

## 3. Прежний BPE08

Словарь 4096 встроен, без переобучения. Проверяем обратимость наtrain и всех 1024 heldout.

In [4]:
tokenizer_started = time.perf_counter()
# Fixed own tokenizer from user run08. No model weights or heldout data embedded.
TOKENIZER_ZLIB_BASE64 = (
    '...'
)
tokenizer_raw = zlib.decompress(base64.b64decode(TOKENIZER_ZLIB_BASE64))
EXPECTED_TOKENIZER_SHA256 = '73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86'
assert sha(tokenizer_raw) == EXPECTED_TOKENIZER_SHA256
TOKENIZER_FILE.write_bytes(tokenizer_raw)
tokenizer = Tokenizer.from_str(tokenizer_raw.decode('utf-8'))
VOCAB = tokenizer.get_vocab_size() + TOKEN_OFFSET
assert VOCAB == TARGET_VOCAB
tokenizer_hash = sha(TOKENIZER_FILE.read_bytes())

# The reversible ByteLevel alphabet, as in tokenizers/src/pre_tokenizers/byte_level.rs.
visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
byte_chars = {b: chr(b) for b in visible}
for index, b in enumerate(b for b in range(256) if b not in byte_chars):
    byte_chars[b] = chr(256 + index)
char_bytes = {char: b for b, char in byte_chars.items()}
assert set(char_bytes) == set(pre_tokenizers.ByteLevel.alphabet())
TOKEN_BYTES = [b'', b'', b''] + [bytes(char_bytes[c] for c in tokenizer.id_to_token(i))
                                for i in range(VOCAB - TOKEN_OFFSET)]
assert all(TOKEN_BYTES[i] for i in range(TOKEN_OFFSET, VOCAB))

def encode(text): return [i + TOKEN_OFFSET for i in tokenizer.encode(text, add_special_tokens=False).ids]
def decode_bytes(ids):
    parts = []
    for i in ids:
        if i == EOS: break
        if i >= TOKEN_OFFSET: parts.append(TOKEN_BYTES[i])
    return b''.join(parts)
def decode(ids): return decode_bytes(ids).decode('utf-8', errors='replace')
def length_stats(values):
    values = sorted(values)
    return {'count': len(values), 'mean': sum(values) / len(values), 'min': values[0],
        'p50': values[int(0.5 * (len(values) - 1))], 'p95': values[int(0.95 * (len(values) - 1))],
        'max': values[-1]}
def audit_tokenization(rows):
    report = {}
    for field in ('question', 'code'):
        byte_lengths, bpe_lengths = [], []
        for row in rows:
            text = row[field]
            ids = encode(text)
            assert decode_bytes(ids) == text.encode('utf-8'), 'Round-trip failed'
            assert tokenizer.decode([i - TOKEN_OFFSET for i in ids], skip_special_tokens=False) == text
            assert all(TOKEN_OFFSET <= i < VOCAB for i in ids)
            assert len(ids) + 2 <= MAX_POSITIONS
            byte_lengths.append(len(text.encode('utf-8')))
            bpe_lengths.append(len(ids))
        report[field] = {'utf8_bytes': length_stats(byte_lengths), 'bpe_tokens': length_stats(bpe_lengths),
            'total_bytes_per_bpe_token': sum(byte_lengths) / sum(bpe_lengths)}
    return report
probes = ['', '    return 1\n', '\n\n\t x\r\n', '  trailing  ',
          'Привет 世界 🧠', 'x = 0.005 + 123456789\n', '<pad><bos><eos>',
          ''.join(chr(i) for i in range(256))]
for text in probes: assert decode_bytes(encode(text)) == text.encode('utf-8')
reloaded_tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))
assert all(reloaded_tokenizer.encode(t).ids == tokenizer.encode(t).ids for t in probes)
result['tokenization'] = {'kind': 'own byte-level BPE', 'loaded_from_experiment': '08', 'tokenizer_training_performed': False, 'target_model_vocabulary': TARGET_VOCAB,
    'actual_model_vocabulary': VOCAB, 'content_id_offset': TOKEN_OFFSET, 'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS},
    'normalizer': None, 'add_prefix_space': False, 'use_regex': True, 'min_frequency': 2,
    'training_corpus': 'only original20000 train questions and codes; no heldout/test',
    'training_corpus_sha256': BASE20K_SHA256, 'tokenizer_sha256': tokenizer_hash,
    'train_round_trip': audit_tokenization(train), 'validation64_round_trip': audit_tokenization(validation),
    'extra192_round_trip': audit_tokenization(validation_extra),
    'new768_round_trip': audit_tokenization(validation_new), 'probe_round_trip_count': len(probes),
    'loading_and_audit_seconds': time.perf_counter() - tokenizer_started}
result['phase'] = 'tokenizer_ready'
save_results()
print('TOKENIZATION', json.dumps(result['tokenization']))
for text in ['def solve():\n    return 1\n', '    ', 'return', '12345']:
    print('TOKENS', repr(text), [(i, repr(TOKEN_BYTES[i])) for i in encode(text)])

def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS], dtype=torch.long) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels


TOKENIZATION {"kind": "own byte-level BPE", "loaded_from_experiment": "08", "tokenizer_training_performed": false, "target_model_vocabulary": 4096, "actual_model_vocabulary": 4096, "content_id_offset": 3, "special_ids": {"pad": 0, "bos": 1, "eos": 2}, "normalizer": null, "add_prefix_space": false, "use_regex": true, "min_frequency": 2, "training_corpus": "only original20000 train questions and codes; no heldout/test", "training_corpus_sha256": "1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e", "tokenizer_sha256": "73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86", "train_round_trip": {"question": {"utf8_bytes": {"count": 100000, "mean": 173.72889, "min": 11, "p50": 159, "p95": 311, "max": 512}, "bpe_tokens": {"count": 100000, "mean": 44.36939, "min": 4, "p50": 40, "p95": 81, "max": 169}, "total_bytes_per_bpe_token": 3.9155122484217157}, "code": {"utf8_bytes": {"count": 100000, "mean": 224.42561, "min": 54, "p50": 206, "p95": 406, "max": 511}, "bpe_token

## 4. Модель и LR

Стандартный Pre-LN Transformer и его маски сохраняются. Заменяем только входную проекцию FFN на `SiLU(gate) * value`, выходную — на680→256. Identity после gated-проекции предотвращает вторую активацию. Dropout остаётся после произведения и после выходной проекции. Encoder fast path для ReLU/GELU выключен для gated-слоёв; KV-cache использует тот же FFN.

Новые FFN получают Xavier-матрицы и bias стандартных Linear. Остальные начальные веса и RNG до первого train-batch совпадают с исходной моделью13. Это сравнение архитектур при одном seed; изменения размерности и dropout скрытого слоя всё равно меняют дальнейшую траекторию обучения.

In [5]:
class SwiGLUInput(nn.Module):
    """Two input-dependent branches: SiLU(gate) * value, before FFN dropout."""
    def __init__(self, width, hidden):
        super().__init__()
        self.gate = nn.Linear(width, hidden)
        self.value = nn.Linear(width, hidden)
    def forward(self, x):
        return F.silu(self.gate(x)) * self.value(x)

def replace_ffn_with_swiglu(transformer):
    # Build the original model first: embeddings/attention/norms/head keep seed42 initialization.
    # Isolated CPU RNG initializes only the new FFNs; training RNG is restored on exit.
    with torch.random.fork_rng(devices=[]):
        torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())
        for layer in list(transformer.encoder.layers) + list(transformer.decoder.layers):
            layer.linear1 = SwiGLUInput(D_MODEL, SWIGLU_DIM)
            layer.linear2 = nn.Linear(SWIGLU_DIM, D_MODEL)
            layer.activation = nn.Identity()  # SiLU and multiplication already happen in linear1.
            if isinstance(layer, nn.TransformerEncoderLayer):
                # Fused encoder supports only ReLU/GELU and would bypass the gated module.
                layer.activation_relu_or_gelu = 0
            for module in [layer.linear1.gate, layer.linear1.value, layer.linear2]:
                nn.init.xavier_uniform_(module.weight)  # same matrix initializer as nn.Transformer

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True, norm_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
        replace_ffn_with_swiglu(self.transformer)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
assert 0 < WARMUP_STEPS < TOTAL_STEPS
def learning_rate_for_update(update, warmup_steps=WARMUP_STEPS, total_steps=SCHEDULE_TOTAL_STEPS,
                             peak_lr=LEARNING_RATE, min_lr=MIN_LEARNING_RATE):
    # Index1 is the first update. Clamp after training to prevent a cosine restart.
    update = min(max(int(update), 1), total_steps)
    if update <= warmup_steps:
        return peak_lr * update / warmup_steps
    progress = (update - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

def learning_rate_multiplier(completed_updates):
    return learning_rate_for_update(completed_updates + 1) / LEARNING_RATE

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=learning_rate_multiplier)
assert scheduler.last_epoch == 0
assert math.isclose(optimizer.param_groups[0]['lr'], LEARNING_RATE / WARMUP_STEPS)
TOKEN_BYTE_LENGTHS = torch.tensor([len(b) for b in TOKEN_BYTES], device=device)
config = {'seed': SEED, 'train_size': len(train), 'batch_size_train': BATCH_SIZE, 'total_steps': TOTAL_STEPS,
    'evaluation_steps': list(EVALUATION_STEPS), 'full_evaluation_steps': list(FULL_EVALUATION_STEPS), 'd_model': D_MODEL, 'encoder_layers': N_LAYERS,
    'decoder_layers': N_LAYERS, 'heads': 4, 'ffn_dim': SWIGLU_DIM, 'relu_reference_ffn_dim': 4 * D_MODEL, 'dropout': 0.1,
    'activation': 'SwiGLU (SiLU gate * linear value)', 'ffn_bias': True, 'norm_first': True, 'lr': LEARNING_RATE, 'optimizer': 'AdamW',
    'weight_decay': optimizer.param_groups[0]['weight_decay'], 'clip_norm': 1.0, 'scheduler': {'name': 'LambdaLR', 'schedule': 'linear_warmup_then_cosine',
        'total_steps': SCHEDULE_TOTAL_STEPS, 'warmup_steps': WARMUP_STEPS,
        'cosine_steps': SCHEDULE_TOTAL_STEPS - WARMUP_STEPS, 'peak_lr': LEARNING_RATE,
        'eta_min': MIN_LEARNING_RATE, 'first_update_lr': LEARNING_RATE / WARMUP_STEPS,
        'restarts': False, 'step_unit': 'optimizer update',
        'indexing': 's=1..T; updateW uses peak; updateT uses floor; scheduler index counts completed updates'},
    'dtype': 'float32', 'tokenizer': 'own byte-level BPE', 'vocab': VOCAB, 'content_id_offset': TOKEN_OFFSET,
    'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS}, 'tokenizer_sha256': tokenizer_hash,
    'max_positions': MAX_POSITIONS, 'generation_cap': GENERATION_CAP,
    'batch_size_inference': INFERENCE_BATCH, 'inference': 'Pre-LN incremental KV cache'}
config['tokenizer_source_experiment'] = '08'
config['training_from_scratch'] = True
config['ffn_initialization'] = 'Original model initialized as13; only FFNs replaced under isolated seed42 CPU RNG, Xavier matrices and nn.Linear biases; no checkpoint loaded'
config['scheduler']['lr_after_original_horizon'] = MIN_LEARNING_RATE
config['ffn_parameter_comparison'] = {'relu_per_layer': 525568, 'swiglu_per_layer': 523856,
    'relu_total': 6312960, 'swiglu_total': 6306112, 'relative_total_change': -6848 / 6312960}
result['config'] = config
result['parameters'] = sum(p.numel() for p in model.parameters())
assert result['parameters'] == 6306112
result['phase'] = 'model_ready'
save_results()
print('PARAMETERS', result['parameters'])
assert tokenizer_hash == result['historical_relu14']['config']['tokenizer_sha256']
assert all(layer.norm_first for layer in list(model.transformer.encoder.layers) + list(model.transformer.decoder.layers))
assert all(layer.activation_relu_or_gelu == 0 for layer in model.transformer.encoder.layers)
assert model.transformer.encoder.norm is not None and model.transformer.decoder.norm is not None


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


PARAMETERS 6306112


In [6]:
class CachedDecoder:
    @torch.inference_mode()
    def __init__(self, model, src, capacity):
        model.eval()
        if not 0 < capacity <= model.positions.num_embeddings:
            raise ValueError('Invalid cache capacity')
        self.model, self.capacity, self.position = model, capacity, 0
        self.batch_size = len(src)
        self.width = model.tokens.embedding_dim
        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads
        self.head_dim = self.width // self.heads
        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).
        self.source_allowed = src.ne(PAD)[:, None, None, :]
        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)
        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))
        self.layers = []
        for layer in model.transformer.decoder.layers:
            if not layer.norm_first:
                raise ValueError('This experiment requires Pre-LN decoder layers')
            for attention in (layer.self_attn, layer.multihead_attn):
                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads
                        or attention.bias_k is not None or attention.bias_v is not None
                        or attention.add_zero_attn):
                    raise ValueError('Unsupported attention configuration')
            attention = layer.multihead_attn
            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],
                                    attention.in_proj_bias[self.width:])
            cross_k, cross_v = projected_kv.chunk(2, dim=-1)
            shape = (len(src), self.heads, capacity, self.head_dim)
            self.layers.append({
                'self_k': memory.new_empty(shape), 'self_v': memory.new_empty(shape),
                'cross_k': self.split_heads(cross_k), 'cross_v': self.split_heads(cross_v),
            })

    def split_heads(self, values):
        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

    def attend(self, attention, q, k, v, allowed):
        # One query at the latest position; all supplied keys are from its past/current.
        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                                dropout_p=0.0, is_causal=False)
        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)
        return attention.out_proj(output)

    @torch.inference_mode()
    def step(self, token_ids):
        if self.position >= self.capacity:
            raise ValueError('Cache capacity exceeded')
        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:
            raise ValueError('One token per batch row is required')
        t = self.position
        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)
        x = x + self.position_vectors[t][None, None, :]
        self.valid[:, t] = token_ids.ne(PAD)
        self_allowed = self.valid[:, None, None, :t + 1]
        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):
            attention = layer.self_attn
            q, k, v = F.linear(layer.norm1(x), attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)
            cache['self_k'][:, :, t:t + 1] = self.split_heads(k)
            cache['self_v'][:, :, t:t + 1] = self.split_heads(v)
            self_output = self.attend(attention, self.split_heads(q),
                cache['self_k'][:, :, :t + 1], cache['self_v'][:, :, :t + 1], self_allowed)
            x = x + layer.dropout1(self_output)

            attention = layer.multihead_attn
            q = F.linear(layer.norm2(x), attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])
            cross_output = self.attend(attention, self.split_heads(q),
                                      cache['cross_k'], cache['cross_v'], self.source_allowed)
            x = x + layer.dropout2(cross_output)
            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x)))))
            x = x + layer.dropout3(feed_forward)

        if self.model.transformer.decoder.norm is not None:
            x = self.model.transformer.decoder.norm(x)
        self.position += 1
        return self.model.head(x[:, 0])

@torch.inference_mode()
def generate_cached(model, src, max_new_tokens=512, max_output_bytes=None):
    model.eval()
    cache = CachedDecoder(model, src, max_new_tokens)
    tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
    done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
    byte_stopped = done.clone()
    output_bytes = torch.zeros(len(src), device=src.device, dtype=torch.long)
    for _ in range(max_new_tokens):
        logits = cache.step(tgt[:, -1])
        logits[:, [PAD, BOS]] = -torch.inf
        token = logits.argmax(-1).masked_fill(done, PAD)
        if max_output_bytes is not None:
            overflow = (~done) & (token >= TOKEN_OFFSET) & (output_bytes + TOKEN_BYTE_LENGTHS[token] > max_output_bytes)
            # Stop BEFORE the whole overflowing BPE token; do not force a different token/EOS.
            byte_stopped |= overflow
            token = token.masked_fill(overflow, PAD)
            done |= overflow
        output_bytes += TOKEN_BYTE_LENGTHS[token]
        done |= token.eq(EOS)
        tgt = torch.cat([tgt, token[:, None]], dim=1)
        if bool(done.all()): break
    return tgt[:, 1:], byte_stopped

@torch.inference_mode()
def check_cached(rows, prefix_limit=64, greedy_cap=16):
    model.eval()
    src = source_batch([r['question'] for r in rows])
    tgt = pad_sequence([torch.tensor([BOS] + encode(r['code']) + [EOS]) for r in rows],
                       batch_first=True, padding_value=PAD).to(device)[:, :prefix_limit]
    memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
    cache = CachedDecoder(model, src, tgt.shape[1])
    positions = {0, 1, 2, 7, 15, 31, tgt.shape[1] - 1}
    checks = []
    for t in range(tgt.shape[1]):
        actual = cache.step(tgt[:, t])
        if t in positions:
            expected = model.head(model.decode_prefix(tgt[:, :t+1], memory, src.eq(PAD))[:, -1])
            checks.append({'position': t, 'max_abs_error': (actual - expected).abs().max().item(),
                'all_close': bool(torch.isclose(actual, expected, atol=LOGIT_ATOL, rtol=LOGIT_RTOL).all())})
    ordinary = model.generate(src, max_new_tokens=greedy_cap)
    cached, _ = generate_cached(model, src, max_new_tokens=greedy_cap)
    return {'positions': checks, 'atol': LOGIT_ATOL, 'rtol': LOGIT_RTOL,
        'all_close': all(c['all_close'] for c in checks),
        'greedy_equal': ordinary.shape == cached.shape and bool(torch.equal(ordinary, cached)),
        'greedy_cap': greedy_cap}


## 5. Контроль генерации и качества

В2k/5k/10k/20k/30k — train64, validation64 и чувствительность к перестановке входных вопросов. В50k/75k/100k дополнительно полный heldout1024 и сравнение с сохранённым ReLU14 на том же шаге. Диагностика сохраняет RNG и режим модели. Проверка обычного decoder против KV выполняется до обучения и на каждом контроле. Программы не выполняются: используется ограниченный AST-интерпретатор арифметики.

In [7]:
class Unsupported(Exception):
    pass

class Undefined(Exception):
    pass

def arithmetic(code):
    try:
        tree = ast.parse(code)
        if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):
            raise Unsupported()
        f = tree.body[0]
        if f.name != 'solve' or f.decorator_list or f.args.args or f.args.posonlyargs or f.args.kwonlyargs or f.args.vararg or f.args.kwarg:
            raise Unsupported()
        if not all((isinstance(s, (ast.Assign, ast.Return)) for s in f.body)):
            raise Unsupported()
        if any((isinstance(n, (ast.Call, ast.Attribute, ast.Subscript, ast.NamedExpr)) for n in ast.walk(f))):
            raise Unsupported()
        env = {}
        visits = 0

        def number(v):
            if type(v) not in (int, float) or not math.isfinite(v) or abs(v) > 1000000000000000.0:
                raise Unsupported()
            return v

        def expr(n):
            nonlocal visits
            visits += 1
            if visits > 2000:
                raise Unsupported()
            if isinstance(n, ast.Constant):
                return number(n.value)
            if isinstance(n, ast.Name):
                if n.id not in env:
                    raise Undefined(n.id)
                return env[n.id]
            if isinstance(n, ast.UnaryOp):
                v = expr(n.operand)
                if isinstance(n.op, ast.USub):
                    return number(-v)
                if isinstance(n.op, ast.UAdd):
                    return v
            if isinstance(n, ast.BinOp):
                a, b = (expr(n.left), expr(n.right))
                if isinstance(n.op, ast.Add):
                    return number(a + b)
                if isinstance(n.op, ast.Sub):
                    return number(a - b)
                if isinstance(n.op, ast.Mult):
                    return number(a * b)
                if isinstance(n.op, ast.Div):
                    return number(a / b)
                if isinstance(n.op, ast.FloorDiv):
                    return number(a // b)
                if isinstance(n.op, ast.Mod):
                    return number(a % b)
                if isinstance(n.op, ast.Pow) and abs(b) <= 12:
                    return number(a ** b)
            raise Unsupported()
        for s in f.body:
            if isinstance(s, ast.Assign):
                if not all((isinstance(n, ast.Name) for n in s.targets)):
                    raise Unsupported()
                v = expr(s.value)
                for n in s.targets:
                    env[n.id] = v
            else:
                return {'status': 'numeric', 'value': expr(s.value)}
        return {'status': 'no_return'}
    except Undefined as e:
        return {'status': 'undefined_name', 'name': str(e)}
    except (SyntaxError, ValueError, RecursionError):
        return {'status': 'syntax_error'}
    except (ZeroDivisionError, OverflowError):
        return {'status': 'arithmetic_error'}
    except Unsupported:
        return {'status': 'unsupported'}

def repetition_properties(code, ids):
    # Heuristics only: repeated token-tail and identical consecutive nonempty lines.
    ids = [i for i in ids if i >= TOKEN_OFFSET]
    tail = 0
    for period in range(1, min(16, len(ids) // 4) + 1):
        motif = ids[-period:]
        copies = 1
        while len(ids) >= (copies + 1) * period and ids[-(copies + 1) * period:-copies * period] == motif:
            copies += 1
        if copies >= 4: tail = max(tail, copies * period)
    longest, run, previous = 0, 0, None
    for line in (line.strip() for line in code.splitlines() if line.strip()):
        run = run + 1 if line == previous else 1
        previous, longest = line, max(longest, run)
    try:
        identifiers = [a.id for a in ast.walk(ast.parse(code)) if isinstance(a, ast.Name)]
        identifier_chars = max(map(len, identifiers), default=0)
    except (SyntaxError, ValueError, RecursionError): identifier_chars = None
    return {'repeated_tail_tokens': tail, 'repeated_tail_32_plus': tail >= 32,
        'max_consecutive_identical_lines': longest, 'consecutive_lines_3_plus': longest >= 3,
        'max_identifier_characters': identifier_chars}

def extra_properties(example):
    numeric = arithmetic(example['generated_code'])
    extra = {'numeric_ast': numeric, **repetition_properties(example['generated_code'], example['generated_token_ids'])}
    if 'reference_code' in example:
        reference = arithmetic(example['reference_code'])
        comparable = numeric['status'] == reference['status'] == 'numeric'
        extra.update(reference_numeric_ast=reference, numeric_ast_comparable=comparable,
            numeric_ast_match=comparable and math.isclose(numeric['value'], reference['value'], rel_tol=1e-7, abs_tol=1e-7))
    return extra

@torch.inference_mode()
def evaluate_loss(rows, source_questions=None):
    model.eval()
    if source_questions is not None: assert len(source_questions) == len(rows)
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        if source_questions is not None: src = source_batch(source_questions[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
            ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    return loss_sum / label_count

@torch.inference_mode()
def check_question_conditioning(rows, correct_loss=None):
    assert len(rows) > 1
    questions = [row['question'] for row in rows]
    rotated = questions[1:] + questions[:1]
    correct = evaluate_loss(rows) if correct_loss is None else correct_loss
    shuffled = evaluate_loss(rows, source_questions=rotated)
    return {'examples': len(rows), 'correct_bpe_loss': correct, 'rotated_questions_bpe_loss': shuffled,
            'rotated_minus_correct': shuffled - correct,
            'scope': 'fixed rotation by1; only source questions change, gold codes/masks/weights unchanged; eval mode; sensitivity, not pass@1'}


def code_properties(code):
    def scoped_nodes(node):
        stack = [node]
        while stack:
            current = stack.pop()
            yield current
            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):
                stack.extend(ast.iter_child_nodes(current))
    try:
        tree = ast.parse(code)
        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'solve']
        has_return = any(isinstance(n, ast.Return) and n.value is not None
            for f in functions for stmt in f.body for n in scoped_nodes(stmt))
        return {'parses_as_python': True, 'defines_solve': bool(functions), 'solve_returns_value': has_return,
                'syntax_error_type': None, 'syntax_error_message': None, 'indentation_error': False}
    except (SyntaxError, ValueError, RecursionError) as error:
        return {'parses_as_python': False, 'defines_solve': False, 'solve_returns_value': False,
                'syntax_error_type': type(error).__name__, 'syntax_error_message': str(error),
                'indentation_error': isinstance(error, IndentationError)}
def inspect_ids(row, ids, byte_stopped=False):
    active = [i for i in ids if i != PAD]
    if EOS in active: active = active[:active.index(EOS) + 1]
    raw = decode_bytes(active)
    try:
        code = raw.decode('utf-8')
        valid_utf8 = True
    except UnicodeDecodeError:
        code, valid_utf8 = raw.decode('utf-8', errors='replace'), False
    eos = EOS in active
    example = {'question': row['question'], 'generated_code': code, 'generated_token_ids': active,
        'generated_tokens_including_eos': len(active), 'generated_utf8_bytes': len(raw), 'valid_utf8': valid_utf8,
        'eos_emitted': eos, 'reached_token_cap': len(active) == GENERATION_CAP,
        'token_cap_without_eos': len(active) == GENERATION_CAP and not eos,
        'byte_budget_stop': bool(byte_stopped),
        'stop_reason': 'eos' if eos else ('byte_budget' if byte_stopped else 'token_cap'),
        **code_properties(code)}
    if 'code' in row:
        example.update({'reference_code': row['code'], 'reference_utf8_bytes': len(row['code'].encode()),
            'reference_bpe_tokens_including_eos': len(encode(row['code'])) + 1,
            'exact_code_match': valid_utf8 and code == row['code'],
            'exact_tokens_with_eos': active == encode(row['code']) + [EOS]})
    return example

@torch.inference_mode()
def inspect_split(name, rows, max_output_bytes=None, with_loss=True):
    model.eval()
    fixed_loss = evaluate_loss(rows) if with_loss else None
    # Warm-up is outside generation timing and never changes weights.
    generate_cached(model, source_batch([rows[0]['question']]), max_new_tokens=8)
    sync()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        tokens, stopped = generate_cached(model, source_batch([r['question'] for r in batch]),
            max_new_tokens=GENERATION_CAP, max_output_bytes=max_output_bytes)
        examples.extend(inspect_ids(row, ids, stop) for row, ids, stop
                        in zip(batch, tokens.cpu().tolist(), stopped.cpu().tolist()))
    sync()
    seconds = time.perf_counter() - started
    extra_started = time.perf_counter()
    for example in examples: example.update(extra_properties(example))
    codes = Counter(e['generated_code'] for e in examples)
    extra_seconds = time.perf_counter() - extra_started
    summary = {'split': name, 'examples': len(rows), 'bpe_loss': fixed_loss, 'seconds': seconds,
        'answers_per_second': len(rows) / seconds,
        'generated_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples),
        'generated_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples),
        'mean_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples) / len(rows),
        'mean_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples) / len(rows),
        'max_new_tokens': GENERATION_CAP, 'max_output_bytes': max_output_bytes,
        'batch_size_inference': INFERENCE_BATCH, 'peak_gpu_memory_mib':
            torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None,
        'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics',
        'extra_diagnostics_seconds_excluded': extra_seconds,
        'unique_generated_code_count': len(codes),
        'most_common_generated_code_count': codes.most_common(1)[0][1],
        'most_common_generated_code_fraction': codes.most_common(1)[0][1] / len(examples),
        'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1',
        'numeric_ast_statuses': dict(Counter(e['numeric_ast']['status'] for e in examples)),
        'numeric_ast_undefined_name_count': sum(e['numeric_ast']['status'] == 'undefined_name' for e in examples),
        'repeated_tail_32_plus_count': sum(e['repeated_tail_32_plus'] for e in examples),
        'consecutive_lines_3_plus_count': sum(e['consecutive_lines_3_plus'] for e in examples),
        'syntax_error_types': dict(Counter(e['syntax_error_type'] for e in examples if e['syntax_error_type']))}
    for metric in ['parses_as_python', 'defines_solve', 'solve_returns_value', 'indentation_error', 'valid_utf8',
                   'eos_emitted', 'reached_token_cap', 'token_cap_without_eos', 'byte_budget_stop']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
    if with_loss:
        summary['numeric_ast_comparable_count'] = sum(e['numeric_ast_comparable'] for e in examples)
        summary['numeric_ast_match_count'] = sum(e['numeric_ast_match'] for e in examples)
        summary['reference_numeric_ast_statuses'] = dict(Counter(e['reference_numeric_ast']['status'] for e in examples))
        for metric in ['exact_code_match', 'exact_tokens_with_eos']:
            summary[metric + '_count'] = sum(e[metric] for e in examples)
    summary['bpe_tokens_per_second'] = summary['generated_tokens_including_eos'] / seconds
    summary['output_bytes_per_second'] = summary['generated_utf8_bytes'] / seconds
    print('DIAGNOSTICS', json.dumps(summary), flush=True)
    for e in examples[:3]: print('EXAMPLE', name, json.dumps(e, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

def save_checkpoint(step):
    state = {'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'steps': step, 'config': config,
        'data_revision': revision, 'train_sha256': train_hash, 'validation_sha256': validation_hash,
        'tokenizer_json': TOKENIZER_FILE.read_text(encoding='utf-8'), 'tokenizer_sha256': tokenizer_hash,
        'python_random_state': random.getstate(), 'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all() if device.type == 'cuda' else [],
        'heldout1024_sha256': data_hash(heldout_all)}
    temporary = LATEST_CHECKPOINT.with_suffix('.pt.tmp')
    torch.save(state, temporary)
    temporary.replace(LATEST_CHECKPOINT)
    result['latest_checkpoint_step'] = step
    if step in FULL_EVALUATION_STEPS:
        name = f'step-{step:05d}.pt'
        target = RUN_DIR / name
        temporary = target.with_suffix('.pt.tmp')
        torch.save(state, temporary)
        temporary.replace(target)
        result['checkpoint_files'][str(step)] = name
    save_results()

def fixed_reference_numeric_audit(examples):
    # Fixed eligibility depends only on reference, never on generated-program success.
    selected = [e for e in examples if e['reference_numeric_ast']['status'] == 'numeric']
    statuses = Counter(e['numeric_ast']['status'] for e in selected)
    agreements = sum(e['numeric_ast_match'] for e in selected)
    pairs = [{'question': e['question'], 'reference_code': e['reference_code']} for e in examples]
    return {'heldout_pairs_total': len(examples), 'reference_numeric_pairs': len(selected),
        'reference_unsupported_pairs': len(examples) - len(selected),
        'questions_and_references_sha256': data_hash(pairs),
        'generated_statuses_on_fixed_reference_subset': dict(statuses),
        'numeric_reference_agreements': agreements,
        'wrong_numeric_count': statuses['numeric'] - agreements,
        'definite_failure_count': sum(n for k, n in statuses.items() if k not in ('numeric', 'unsupported')),
        'unresolved_generated_unsupported_count': statuses['unsupported'],
        'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}

@contextlib.contextmanager
def preserve_evaluation_state():
    python_rng, cpu_rng = random.getstate(), torch.get_rng_state().clone()
    cuda_rng = torch.cuda.get_rng_state_all() if device.type == 'cuda' else []
    was_training = model.training
    try:
        yield
    finally:
        random.setstate(python_rng)
        torch.set_rng_state(cpu_rng)
        if device.type == 'cuda': torch.cuda.set_rng_state_all(cuda_rng)
        model.train(was_training)

def combine_heldout(groups):
    examples = [e for group in groups for e in group['examples']]
    counts = [sum(e['reference_bpe_tokens_including_eos'] for e in g['examples']) for g in groups]
    weighted_loss = sum(g['summary']['bpe_loss'] * n for g, n in zip(groups, counts)) / sum(counts)
    return {'examples': len(examples), 'bpe_loss': weighted_loss,
        'numeric_audit': fixed_reference_numeric_audit(examples),
        'parses_as_python_count': sum(e['parses_as_python'] for e in examples),
        'eos_emitted_count': sum(e['eos_emitted'] for e in examples),
        'token_cap_without_eos_count': sum(e['token_cap_without_eos'] for e in examples),
        'undefined_name_count': sum(e['numeric_ast']['status'] == 'undefined_name' for e in examples),
        'generation_seconds_sum': sum(g['summary']['seconds'] for g in groups),
        'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}

@torch.inference_mode()
def evaluate_control(step):
    with preserve_evaluation_state():
        parity = check_cached(validation[:3])
        if not parity['all_close'] or not parity['greedy_equal']:
            raise RuntimeError('KV parity failed; download results instead of continuing')
        full = step in FULL_EVALUATION_STEPS
        control = {'cache_check': parity, 'full_heldout_evaluated': full}
        result['diagnostics'][str(step)] = control
        groups = [('train64', train_diagnostic), ('validation64', validation)]
        if full:
            groups += [('validation192_extra', validation_extra), ('validation768_new', validation_new)]
        for name, rows in groups:
            control[name] = inspect_split(name, rows)
            save_results()
        control['validation64_conditioning'] = check_question_conditioning(validation,
            correct_loss=control['validation64']['summary']['bpe_loss'])
        if full:
            original = [control['validation64'], control['validation192_extra']]
            control['heldout256'] = combine_heldout(original)
            control['heldout1024'] = combine_heldout(original + [control['validation768_new']])
            prior = result['historical_relu14']['controls'][str(step)]
            for name in ['heldout256', 'heldout1024']:
                a, b = control[name]['numeric_audit'], prior[name]['numeric_audit']
                assert a['questions_and_references_sha256'] == b['questions_and_references_sha256']
                assert a['reference_numeric_pairs'] == b['reference_numeric_pairs']
            control['comparison_with_saved_relu14'] = {
                'provenance': 'Historical T4 run14 ReLU versus current SwiGLU at the same update; no baseline retraining',
                'heldout1024_CE': {'saved_relu14': prior['heldout1024']['bpe_loss'], 'current_swiglu15': control['heldout1024']['bpe_loss']},
                'reference_agreements_out_of_888': {'saved_relu14': prior['heldout1024']['numeric_audit']['numeric_reference_agreements'],
                    'current_swiglu15': control['heldout1024']['numeric_audit']['numeric_reference_agreements']},
                'eos_out_of_1024': {'saved_relu14': prior['heldout1024']['eos_emitted_count'], 'current_swiglu15': control['heldout1024']['eos_emitted_count']},
                'undefined_names_out_of_1024': {'saved_relu14': prior['heldout1024']['undefined_name_count'], 'current_swiglu15': control['heldout1024']['undefined_name_count']}}
            print('SAVED_RELU14_CURRENT_SWIGLU15', json.dumps({'step': step, **control['comparison_with_saved_relu14']}), flush=True)
        else:
            print('EARLY_CONTROL', step, json.dumps(control['validation64']['summary']), flush=True)
        save_results()
    return control

# A small implementation check, not an overfit experiment or a quality measurement.
with preserve_evaluation_state():
    result['initial_cache_check'] = check_cached(validation[:3])
    assert result['initial_cache_check']['all_close'] and result['initial_cache_check']['greedy_equal']
    print('INITIAL_CACHE_CHECK', json.dumps(result['initial_cache_check']))
save_results()


INITIAL_CACHE_CHECK {"positions": [{"position": 0, "max_abs_error": 4.76837158203125e-07, "all_close": true}, {"position": 1, "max_abs_error": 4.76837158203125e-07, "all_close": true}, {"position": 2, "max_abs_error": 5.364418029785156e-07, "all_close": true}, {"position": 7, "max_abs_error": 5.364418029785156e-07, "all_close": true}, {"position": 15, "max_abs_error": 3.5762786865234375e-07, "all_close": true}, {"position": 31, "max_abs_error": 3.5762786865234375e-07, "all_close": true}, {"position": 63, "max_abs_error": 7.152557373046875e-07, "all_close": true}], "atol": 0.0001, "rtol": 0.0001, "all_close": true, "greedy_equal": true, "greedy_cap": 16}


## 6. Обучение с нуля до 100000 шагов

Latest сохраняется каждые 1000 шагов, отдельные checkpoints — 50k/75k/100k. Не запускай ячейку повторно в той же сессии: при прерывании сначала сохрани ZIP. Время обучения исключает диагностику, логирование и сохранение checkpoints; полное активное время тоже записывается.

In [8]:
assert not result.get('training_started') and not optimizer.state and result['actual_steps'] == 0, 'Train уже запускался; сохрани файлы, не запускай ячейку повторно'
assert scheduler.last_epoch == 0 and math.isclose(optimizer.param_groups[0]['lr'], LEARNING_RATE / WARMUP_STEPS)
assert result['initial_cache_check']['all_close'] and result['initial_cache_check']['greedy_equal']
result['training_started'] = True
assert next(model.parameters()).dtype == torch.float32
result['phase'] = 'training'
result['training_timer_scope'] = 'batch encoding/construction, transfers, forward/backward, clipping, AdamW, scheduler, block-end CUDA sync'
result['training_log'].append({'step': 0, 'train_batch_loss': None,
    'validation_loss_16': evaluate_loss(validation[:16]),
    'lr_next': optimizer.param_groups[0]['lr'], 'training_seconds': 0.0})
save_results()
run_started = time.perf_counter()
training_seconds = 0.0
completed_steps = 0
sync()
block_started, block_active = time.perf_counter(), True
try:
    for step in range(1, TOTAL_STEPS + 1):
        model.train()
        src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradient_norm_before_clip = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        lr_used = optimizer.param_groups[0]['lr']
        optimizer.step()
        scheduler.step()
        completed_steps = step
        if step == 1 or step % LOG_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
            sync()
            training_seconds += time.perf_counter() - block_started
            block_active = False
            result['actual_steps'] = completed_steps
            result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
            train_loss, val_loss = loss.item(), evaluate_loss(validation[:16])
            if not math.isfinite(train_loss) or not math.isfinite(val_loss):
                raise FloatingPointError('Non-finite loss; last finite checkpoint is kept')
            entry = {'step': step, 'train_batch_loss': train_loss, 'validation_loss_16': val_loss,
                'gradient_norm_before_clip': float(gradient_norm_before_clip),
                'lr_used': lr_used, 'lr_next': optimizer.param_groups[0]['lr'],
                'training_seconds': training_seconds, 'updates_per_second': step / training_seconds,
                'mean_pair_presentations': step * BATCH_SIZE / len(train)}
            result['training_log'].append(entry)
            print('LOSS', json.dumps(entry), flush=True)
            save_results()
            if step in EVALUATION_STEPS:
                evaluate_control(step)
            if step % CHECKPOINT_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
                save_checkpoint(completed_steps)
                print('CHECKPOINT', completed_steps, flush=True)
            sync()
            if step < TOTAL_STEPS:
                block_started, block_active = time.perf_counter(), True
    result['run_finished'] = True
    result['phase'] = 'training_finished'
except BaseException as error:
    result['phase'] = 'training_stopped'
    result['training_error'] = {'type': type(error).__name__, 'message': str(error)}
    raise
finally:
    sync()
    if block_active: training_seconds += time.perf_counter() - block_started
    result['actual_steps'] = completed_steps
    result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
    result['active_wall_seconds_including_diagnostics_and_checkpoints'] = time.perf_counter() - run_started
    if result.get('training_error', {}).get('type') != 'FloatingPointError':
        save_checkpoint(completed_steps)
    save_results()


LOSS {"step": 1, "train_batch_loss": 8.478312492370605, "validation_loss_16": 8.431752288114343, "gradient_norm_before_clip": 1.818556547164917, "lr_used": 1e-06, "lr_next": 2e-06, "training_seconds": 0.25609723000002305, "updates_per_second": 3.904766951207985, "mean_pair_presentations": 8e-05}
LOSS {"step": 50, "train_batch_loss": 8.369990348815918, "validation_loss_16": 8.255523406155968, "gradient_norm_before_clip": 1.2252432107925415, "lr_used": 5e-05, "lr_next": 5.1000000000000006e-05, "training_seconds": 1.3804916420000382, "updates_per_second": 36.21898059995543, "mean_pair_presentations": 0.004}
LOSS {"step": 100, "train_batch_loss": 7.67277717590332, "validation_loss_16": 7.5344461117728185, "gradient_norm_before_clip": 1.4931755065917969, "lr_used": 0.0001, "lr_next": 0.000101, "training_seconds": 2.4404063050000673, "updates_per_second": 40.97678316725921, "mean_pair_presentations": 0.008}
LOSS {"step": 150, "train_batch_loss": 4.835009574890137, "validation_loss_16": 5.110

## 7. Итоговый аудит и прежний byte512-контроль

In [9]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'final_validation'
result['final_validation'] = {'validation64_byte512': inspect_split('validation64_byte512', validation,
    max_output_bytes=BYTE_BUDGET)}
control = result['diagnostics'][str(TOTAL_STEPS)]
result['fixed_reference_numeric_audit'] = control['heldout256']['numeric_audit']
result['expanded_heldout_numeric_audit'] = control['heldout1024']['numeric_audit']
result['final_heldout_comparison_with_saved_relu14'] = control['comparison_with_saved_relu14']
result['phase'] = 'validation_finished'
save_results()


DIAGNOSTICS {"split": "validation64_byte512", "examples": 64, "bpe_loss": 0.8647585702262776, "seconds": 0.6679276570002912, "answers_per_second": 95.81876020440353, "generated_tokens_including_eos": 3690, "generated_utf8_bytes": 12983, "mean_tokens_including_eos": 57.65625, "mean_utf8_bytes": 202.859375, "max_new_tokens": 512, "max_output_bytes": 512, "batch_size_inference": 32, "peak_gpu_memory_mib": 209.70947265625, "timer_scope": "same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics", "extra_diagnostics_seconds_excluded": 0.019527431999904366, "unique_generated_code_count": 64, "most_common_generated_code_count": 1, "most_common_generated_code_fraction": 0.015625, "numeric_ast_scope": "restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1", "numeric_ast_statuses": {"unsupported": 4, "numeric

## 8. Полный test1919

Прежний порядок и таймер encode/transfers/encoder/KV/CPU decode/AST. Подготовка/loss/warmup исключены; официальная схема таймера неизвестна.

In [10]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'test_data'
save_results()
test_file = cache / 'test.parquet'
if not test_file.exists(): urlretrieve(f'{base_url}/data/test/test.parquet', test_file)
assert sha(test_file.read_bytes()) == '8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a'
test_questions = pq.read_table(test_file, columns=['question'])['question'].to_pylist()
assert len(test_questions) == 1919
assert all(len(encode(q)) + 2 <= MAX_POSITIONS for q in test_questions)
test_rows = [{'question': q} for q in test_questions]
result['test_data'] = {'rows': len(test_rows), 'questions_sha256': data_hash(test_rows),
    'file_sha256': sha(test_file.read_bytes()), 'row_order': 'unchanged parquet order', 'dataset_revision': revision}
result['phase'] = 'test_generation'
save_results()
test_output = inspect_split('test1919', test_rows, with_loss=False)
TEST_GENERATIONS = RUN_DIR / 'test-generations.json'
write_json(TEST_GENERATIONS, test_output['examples'])
result['test'] = {'summary': test_output['summary'], 'examples_file': TEST_GENERATIONS.name}
result['test']['summary']['within_180_seconds'] = test_output['summary']['seconds'] <= 180
result['test_evaluated'] = True
result['phase'] = 'complete'
save_results()
print('TEST_SUMMARY', json.dumps(result['test']['summary']), flush=True)


DIAGNOSTICS {"split": "test1919", "examples": 1919, "bpe_loss": null, "seconds": 36.6384470109997, "answers_per_second": 52.37667413752205, "generated_tokens_including_eos": 123298, "generated_utf8_bytes": 417996, "mean_tokens_including_eos": 64.25117248566961, "mean_utf8_bytes": 217.8196977592496, "max_new_tokens": 512, "max_output_bytes": null, "batch_size_inference": 32, "peak_gpu_memory_mib": 223.74267578125, "timer_scope": "same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics", "extra_diagnostics_seconds_excluded": 0.4066479309999522, "unique_generated_code_count": 1909, "most_common_generated_code_count": 3, "most_common_generated_code_fraction": 0.001563314226159458, "numeric_ast_scope": "restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1", "numeric_ast_statuses": {"numeric": 1657, "un

## 9. Скачать результаты, включая частичный запуск

ZIP: results.json, tokenizer.json, latest.pt, выбранные checkpoints, test-generations.json при завершённом test, README и SHA256-manifest. Ячейка только упаковывает и скачивает уже доступные файлы. После полного запуска также скачай IPYNB с outputs через меню Colab.

In [11]:
# This cell never trains, generates or restarts anything.
assert RESULTS_FILE.exists(), 'Сначала должны появиться результаты запуска'
result['archive_prepared'] = True
save_results()
README_FILE = RUN_DIR / 'README.txt'
README_FILE.write_text(
    'Experiment15: fresh100000 updates; ReLU FFN1024 replaced by biased SwiGLU680 in encoder and decoder; otherwise same100k data/BPE/Pre-LN/AdamW/batch8. Warmup1000, cosine to50000, floor0.0001 afterward.\n'
    'results.json contains early64 controls, full1024 controls50000/75000/100000, same-update saved ReLU14 summaries, source hash, phase and actual update count.\n'
    'tokenizer.json has4093 content IDs; model adds offset3 and PAD/BOS/EOS0/1/2.\n'
    'latest.pt and step-*.pt include model, optimizer, scheduler, tokenizer JSON, data hashes and RNG.\n'
    'test-generations.json exists only after complete test. No exec/eval/calls/imports; restricted numeric AST checks are not official pass@1.\n'
    'For analysis return this ZIP and the IPYNB downloaded with outputs.\n', encoding='utf-8')
files_to_pack = sorted(p for p in RUN_DIR.iterdir() if p.is_file()
                       and p.name != 'manifest.json' and not p.name.endswith('.tmp'))
manifest = {'experiment': result['experiment'], 'actual_steps': result['actual_steps'], 'phase': result['phase'],
    'files': [{'name': p.name, 'bytes': p.stat().st_size, 'sha256': sha(p.read_bytes())} for p in files_to_pack]}
write_json(RUN_DIR / 'manifest.json', manifest)
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in files_to_pack + [RUN_DIR / 'manifest.json']: archive.write(p, arcname=p.name)
print('ARCHIVE', str(ARCHIVE_FILE), ARCHIVE_FILE.stat().st_size, 'bytes', '| steps:', result['actual_steps'])
from google.colab import files
files.download(str(ARCHIVE_FILE))


ARCHIVE hw02-swiglu-preln-bpe4k-100k-heldout1024-results.zip 265392601 bytes | steps: 100000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>